# MobileFaceNet (tflite) → CoreML (.mlpackage) — TEK SEFERLİK dönüşüm

Android'in `mobilefacenet.tflite` modelini (192-dim embedding) iOS için CoreML `.mlpackage`'a çevirir.
**Bu yalnızca 1 KEZ çalışır.** Çıktı `MobileFaceNet.mlpackage` repoya commit edilir; Xcode onu her build'de
yalnızca *derler* (coremltools'u TEKRAR çalıştırmaz). Per-build maliyet SIFIR.

## Sözleşme (iOS `FaceEmbedder.swift` bununla uyumlu)
- **Girdi:** 112×112 RGB **image** (`image`). Normalizasyon `(p-127.5)/128` modele gömülü (scale/bias).
- **Çıktı:** 192-dim ham embedding (`embedding`). L2-normalize uygulamada yapılır (Android paritesi).

## Adımlar
1. Üstten **Runtime → Run all** (veya hücre hücre).
2. İstendiğinde `mobilefacenet.tflite` dosyasını yükleyin
   (`src/VerifyBlind.Android/app/src/main/assets/mobilefacenet.tflite`).
3. Parite hücresi cosine ≈ 0.999 yazmalı (dönüşüm sadık demektir).
4. İnen `MobileFaceNet.mlpackage.zip`'i açın → `MobileFaceNet.mlpackage`'ı
   `src/VerifyBlind.iOS/Resources/` içine koyup commit edin.

Zincir: `tflite → onnx (tf2onnx) → torch (onnx2torch) → coreml (coremltools)`.

In [ ]:
# 1) Bağımlılıklar (pinli — kurulum kırılırsa pinleri gevşetin)
!pip -q install "coremltools==8.1" "tensorflow==2.15.0" "tf2onnx==1.16.1" \
    "onnx==1.16.0" "onnx2torch==1.5.15" "torch==2.2.2" pillow numpy

In [ ]:
# 2) mobilefacenet.tflite yükle
from google.colab import files
print('mobilefacenet.tflite seçin...')
up = files.upload()
TFLITE = next(iter(up.keys()))
print('Yüklendi:', TFLITE)

In [ ]:
# 3) tflite → onnx
!python -m tf2onnx.convert --tflite "$TFLITE" --output mobilefacenet.onnx --opset 13
print('onnx üretildi.')

In [ ]:
# 4) onnx → torch ve giriş şeklini tespit et
import onnx, torch, numpy as np
from onnx2torch import convert

onnx_model = onnx.load('mobilefacenet.onnx')
torch_model = convert(onnx_model).eval()

# tf2onnx tflite NHWC'yi genelde NCHW'ye çevirir → (1,3,112,112) bekleriz.
inp = onnx_model.graph.input[0]
shape = [d.dim_value for d in inp.type.tensor_type.shape.dim]
print('onnx input shape:', shape)
example = torch.rand(1, 3, 112, 112)
traced = torch.jit.trace(torch_model, example)
out = traced(example)
print('torch output shape:', tuple(out.shape))

In [ ]:
# 5) torch → CoreML (112x112 RGB image input, normalizasyon gömülü, 192-dim çıktı)
import coremltools as ct

scale = 1.0 / 128.0
bias = [-127.5 / 128.0, -127.5 / 128.0, -127.5 / 128.0]

mlmodel = ct.convert(
    traced,
    inputs=[ct.ImageType(name='image', shape=(1, 3, 112, 112),
                         scale=scale, bias=bias, color_layout=ct.colorlayout.RGB)],
    outputs=[ct.TensorType(name='embedding')],
    minimum_deployment_target=ct.target.iOS16,
    convert_to='mlprogram',
)
mlmodel.short_description = 'MobileFaceNet 112x112 → 192-dim face embedding (Android parity).'
mlmodel.save('MobileFaceNet.mlpackage')
print('CoreML kaydedildi: MobileFaceNet.mlpackage')

In [ ]:
# 6) PARİTE: tflite ve CoreML embedding'leri AYNI mı? (cosine ≈ 0.999 olmalı)
import numpy as np, tensorflow as tf
from PIL import Image

# Deterministik örnek görüntü (112x112 RGB)
rng = np.random.default_rng(42)
img_u8 = rng.integers(0, 256, size=(112, 112, 3), dtype=np.uint8)
pil = Image.fromarray(img_u8, 'RGB')

# --- tflite yolu (Android: (x-127.5)/128, NHWC) ---
interp = tf.lite.Interpreter(model_path=TFLITE)
interp.allocate_tensors()
ind = interp.get_input_details()[0]
outd = interp.get_output_details()[0]
x = ((img_u8.astype(np.float32) - 127.5) / 128.0)[None, ...]  # NHWC
if list(ind['shape'])[1] == 3:  # model NCHW isterse
    x = np.transpose(x, (0, 3, 1, 2))
interp.set_tensor(ind['index'], x)
interp.invoke()
emb_tflite = interp.get_tensor(outd['index']).reshape(-1)

# --- CoreML yolu (image input, normalizasyon gömülü) ---
pred = mlmodel.predict({'image': pil})
emb_coreml = np.array(list(pred.values())[0]).reshape(-1)

def l2(v):
    n = np.linalg.norm(v)
    return v / n if n > 0 else v

cos = float(np.dot(l2(emb_tflite), l2(emb_coreml)))
print('embedding dim:', emb_tflite.shape, emb_coreml.shape)
print('PARİTE cosine =', round(cos, 5), '(≥0.99 ise dönüşüm sadık)')

In [ ]:
# 7) .mlpackage'ı zip'le ve indir → açıp Resources/'a koyup commit edin
import shutil
shutil.make_archive('MobileFaceNet.mlpackage', 'zip', 'MobileFaceNet.mlpackage')
from google.colab import files
files.download('MobileFaceNet.mlpackage.zip')

## Son adım (repoda)
1. İnen `MobileFaceNet.mlpackage.zip`'i açın.
2. Çıkan `MobileFaceNet.mlpackage` klasörünü `src/VerifyBlind.iOS/Resources/` içine kopyalayın.
3. Commit + push (`dev`). Xcodegen `Resources/` zaten source path → otomatik bundle'lanır.
4. Sonraki build'de `FaceEmbedder` modeli bulur → canlı % aktif olur (önce % gizliydi).